In [1]:
%matplotlib qt
import mne

#mne.viz.set_3d_backend('pyvistaqt')
from mne.coreg import Coregistration
from mne.io import read_info


import numpy as np
#%matplotlib qt
import matplotlib
#matplotlib.use('qt5agg')  # Or any other backend you want to use

import matplotlib.pyplot as plt

import pandas as pd 
import os
from os.path import join as pathjoin
from pathlib import Path

import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from time import time

from autoreject import AutoReject

import glob

import psutil
import gc
from time import time

mne.set_log_level('INFO')

import json

from mne.channels import read_dig_polhemus_isotrak  # Función para leer archivos .pos

import re
# from mne.minimum_norm import apply_inverse, make_inverse_operator
#este codigo lo dejo comentado para acostumbrarme a su suso

import brainiak
from brainiak.isc import isc 

import statsmodels
from statsmodels.stats.multitest import multipletests

import pickle

In [2]:
try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "event"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")
    
    
with open(datadir / f"subjects_remove_{modality}.pkl", "rb") as f:
    subjects_remove = pickle.load(f)

print(type(subjects_remove))   # <class 'list'>
print(subjects_remove)




📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_event
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\ICA_event
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_event
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_event\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_event\fwd
inverse_pat

In [3]:
#subjects = subj[:9]

subjects = []

# Recorremos cada subdirectorio en la carpeta base
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'
        
    if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
        # Añadimos el nombre del sujeto a la lista
        subjects.append(subdirectorio.name)

print(subjects)

filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")


['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1017', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089'

In [4]:
def print5(*args):
    print(*(f"{x:.5f}" if isinstance(x, float) else x for x in args))

# Leer un archivo de evoked_block para extraer condiciones y canales
subj = subjects[0]
if filtering==True:
    evoked_file = evoked_path / f"{subj}_evokeds_{filter_name}_{layer_script}-ave.fif"
    # filter_applied=True
else:
    evoked_file = evoked_path / f"{subj}_evokeds_{layer_script}-ave.fif"

# Cargar los evokeds del sujeto
evokeds = mne.read_evokeds(evoked_file)

# Extraer nombres de condiciones (evoked.comment)
conditions = [evk.comment for evk in evokeds]
print("Condiciones encontradas en evoked_block:", conditions)

# ==============================
# ✅ Obtener canales MAG desde el evoked (no CSV)
# ==============================

# Usa el primer evoked para leer info de canales
ev0 = evokeds[0]

# obtener nombres MAG
channels_mag = mne.pick_info(ev0.info, sel=None, )['ch_names']


print("\nCanales MAG detectados desde el Evoked:")
print5(channels_mag)

# obtener índices MAG
indices_mag = mne.pick_types(ev0.info, meg='mag', eeg=False, exclude=[])

print("Índices MAG en el Evoked:", indices_mag)
print(f"Número de canales MAG detectados: {len(indices_mag)}")

del evokeds, ev0  # liberar memoria


filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event\sub-V1001_evokeds_filt_1-40_event-ave.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms (woorden_RC_neg)
        0 CTF compensation matrices available
        nave = 89 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
    Found the data of interest:
        t =       0.00 ...    8000.00 ms (woorden_RC_neg_question_hit)
        0 CTF compensation matrices available
        nave = 15 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
    Found the data of interest:
        t =       0.00 ...    8000.00 ms (woorden_RC_neg_question_incorrect)
        0 CTF compensation matrices available
        nave = 5 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline corr

In [5]:
valid_arrays = []
valid_subjects = []

n_subjects=len(range(0,len(subjects)))
# Condiciones
print(f"number of subjects: {n_subjects}")
# Diccionario maestro donde guardaremos TODO
dict_all = {}

for cond in conditions:
    
    valid_arrays = []
    valid_subjects = []

    for i in range(n_subjects):
       
        subj = subjects[i]
        if subj in subjects_remove:
            print(f" ❌ {subj} está en la lista de sujetos a eliminar, se omite.")
            continue
        if filtering:
            file_path = evoked_path / f"{subj}_evokeds_{filter_name}_{layer_script}-ave.fif"
            filter_applied=True
        else:
            file_path = evoked_path / f"{subj}_evokeds_{layer_script}-ave.fif"

        try:
            evokeds = mne.read_evokeds(file_path)
            # encontrar el evoked correcto para esta condición
            evoked = [e for e in evokeds if e.comment == cond][0]
            if "channels_mag" not in dict_all:
                tmp_ev = evoked.copy().pick("mag", exclude="bads")  # misma lógica que usas para data
                dict_all["channels_mag"] = tmp_ev.info['ch_names']
            data_subj = evoked.copy().pick("mag", exclude="bads").data  # (n_sensors, n_times)
            data_subj_swapped = data_subj.T  # (n_times, n_sensors)

            if np.all(data_subj_swapped == 0) or np.isnan(data_subj_swapped).all():
                print(f"{subj} tiene solo ceros o NaNs, se omite.")
                continue

            valid_arrays.append(data_subj_swapped)
            valid_subjects.append(subj)
            evoked_times = evoked.times  # guardar los tiempos del último sujeto leído
            del evokeds, evoked, data_subj, data_subj_swapped  # liberar memoria

        except Exception as e:
            print(f"Error con {subj}, condición {cond}: {e}")
            continue

    array = np.stack(valid_arrays, axis=2)  # (n_times, n_sensors, n_subjects)
    print(f"Array ISC {cond} creado con forma {array.shape} ({len(valid_subjects)} sujetos válidos)")

    iscs = isc(data=array, pairwise=False, summary_statistic=None, tolerate_nans=True)
    iscs_statistics = brainiak.isc.compute_summary_statistic(iscs, summary_statistic='mean', axis=0)

    iscs_bootstrap, ci, p, distribution = brainiak.isc.bootstrap_isc(
        iscs, pairwise=False, summary_statistic='mean', 
        n_bootstraps=1000, ci_percentile=95, side='right', random_state=None
    )

    threshold = 0.05
    significant_channels = np.where(p < threshold)[0]
    num_significant_channels = len(significant_channels)

    _, p_adjusted, _, _ = multipletests(p, method='fdr_bh')
    significant_channels_adjusted = np.where(p_adjusted < threshold)[0]
    num_significant_channels_adjusted = len(significant_channels_adjusted)

    # Mapear índices → nombres de canales
    significant_channels_names = [dict_all["channels_mag"][idx] for idx in significant_channels]
    significant_channels_adjusted_names = [dict_all["channels_mag"][idx] for idx in significant_channels_adjusted]
    # ✅ Crear sub-diccionario por condición
    dict_all[f"dict_isc_{cond}"] = {
        "iscs": iscs,
        "iscs_statistics": iscs_statistics,
        "iscs_bootstrap": iscs_bootstrap,
        "ci": ci,
        "p": p,
        "distribution": distribution,
        "significant_channels": significant_channels,
        "significant_channels_names": significant_channels_names,      # ✅ nuevos nombres
        "num_significant_channels": num_significant_channels,
        "p_adjusted": p_adjusted,
        "significant_channels_adjusted": significant_channels_adjusted,
        "significant_channels_adjusted_names": significant_channels_adjusted_names,   # ✅ nuevos nombres
        "num_significant_channels_adjusted": num_significant_channels_adjusted,
        "valid_subjects": valid_subjects,
        "channels_mag": dict_all["channels_mag"],   # todos los nombres MAG
        "times": evoked_times
    }

# ✅ Guardar un solo archivo con TODO
if filtering==True and filter_applied==True:
    pickle_path = ISC_path / f"ISC_results_{filter_name}_{layer_script}.pkl"
elif filtering==False:
    pickle_path = ISC_path / f"ISC_results_{layer_script}.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(dict_all, f)

print(f"\n✅ Diccionario maestro guardado en: {pickle_path}")
print(f"Contiene {len(dict_all)} condiciones: {list(dict_all.keys())}")

    

number of subjects: 102
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event\sub-V1001_evokeds_filt_1-40_event-ave.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms (woorden_RC_neg)
        0 CTF compensation matrices available
        nave = 89 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
    Found the data of interest:
        t =       0.00 ...    8000.00 ms (woorden_RC_neg_question_hit)
        0 CTF compensation matrices available
        nave = 15 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
    Found the data of interest:
        t =       0.00 ...    8000.00 ms (woorden_RC_neg_question_incorrect)
        0 CTF compensation matrices available
        nave = 5 - aspect type = 100
No projector specified for this dataset. Please consider the method self.ad